# Code mẫu — Encoder–decoder

Model seq2seq đọc trọn chuỗi nguồn, rồi sinh chuỗi đích từng token một, dựa trên phần đã sinh. Các ví dụ dưới đây kiểm ba chuyện: đầu vào của decoder được dựng ra sao, decoder đọc chuỗi nguồn thế nào, và vì sao loss lúc train có thể đẹp hơn nhiều so với chuỗi model tự sinh.

## Ví dụ 1: Decoder nhận token đúng của bước trước

Nhiệm vụ là đảo ngược một chuỗi ký hiệu. Có hai chuỗi nguồn là `A B` và `C`. Đích tương ứng là `B A EOS` và `C EOS`. PAD chỉ dùng để ghép hai chuỗi thành một batch. Token mở đầu cho decoder là `BOS`, có ID bằng 1.

In [ ]:
import torch

torch.set_num_threads(1)
names = ["PAD", "BOS", "EOS", "A", "B", "C"]
PAD, START = 0, 1
source = torch.tensor([[3, 4], [5, 0]])
target = torch.tensor([[4, 3, 2], [5, 2, 0]])
decoder_input = torch.full_like(target, PAD)
decoder_input[:, 0] = START
decoder_input[:, 1:] = target[:, :-1]
labels = target.masked_fill(target.eq(PAD), -100)
for x, y, z in zip(source.tolist(), decoder_input.tolist(), labels.tolist()):
    print("nguồn:", [names[i] for i in x])
    print("decoder:", [names[i] for i in y])
    print("label:", [names[i] if i != -100 else "bỏ" for i in z])
print("số token đích được chấm:", int(labels.ne(-100).sum()))
assert torch.equal(decoder_input[:, 1:], target[:, :-1])
assert decoder_input[:, 0].eq(START).all()
assert labels[:, -1].tolist() == [2, -100]

nguồn: ['A', 'B']
decoder: ['BOS', 'B', 'A']
label: ['B', 'A', 'EOS']
nguồn: ['C', 'PAD']
decoder: ['BOS', 'C', 'EOS']
label: ['C', 'EOS', 'bỏ']
số token đích được chấm: 5


Đầu vào của decoder là chuỗi đích lùi sang phải một ô, với `BOS` ở đầu. Với nguồn `A B`, bước đầu nhận `BOS` và phải đoán `B`. Bước cuối nhận `A` và phải đoán `EOS`. Cách cho decoder nhận token đúng của bước trước như vậy gọi là teacher forcing.

Với nguồn `C`, vị trí cuối của đầu vào là `EOS`, nhưng nhãn ở đó là PAD nên bị bỏ, không góp vào loss. Tổng cộng có 5 token đích được chấm. Token mở đầu phải lấy đúng theo cấu hình của checkpoint đang dùng. Không phải model nào cũng dùng ID 1. Với Transformer, ngoài việc lùi nhãn, decoder còn cần causal mask để không nhìn trước token tương lai.

## Ví dụ 2: Hai query đích đọc ba key nguồn, trong đó có một PAD

Ở bước cross-attention, mỗi vị trí của decoder là một query, và đọc các vị trí của chuỗi nguồn. Query thứ nhất nghiêng về trục đầu, query thứ hai nghiêng về trục sau. Nguồn có hai value thật là `(2, 0)` và `(0, 2)`, cộng một ô PAD có value `(99, 99)`. Giá trị lớn này để thấy rõ chuyện gì xảy ra nếu quên mask.

In [ ]:
import numpy as np

queries = np.array([[[1., 0.], [0., 1.]]])
keys = np.array([[[1., 0.], [0., 1.], [9., 9.]]])
values = np.array([[[2., 0.], [0., 2.], [99., 99.]]])
source_keep = np.array([[True, True, False]])

def cross_attention(queries, keys, values, source_keep):
    if not source_keep.any(axis=1).all():
        raise ValueError("Nguồn không có token thật")
    scores = queries @ keys.swapaxes(-1, -2) / np.sqrt(keys.shape[-1])
    scores = np.where(source_keep[:, None, :], scores, -np.inf)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights, weights @ values

weights, context = cross_attention(queries, keys, values, source_keep)
_, wrong_context = cross_attention(queries, keys, values, np.ones_like(source_keep))
print("shape Q/K/weights:", queries.shape, keys.shape, weights.shape)
print("weights:", weights.round(3).tolist())
print("context có mask:", context.round(3).tolist())
print("context quên mask:", wrong_context.round(3).tolist())
assert np.allclose(weights.sum(axis=-1), 1)
assert np.all(weights[:, :, 2] == 0)
_, without_pad = cross_attention(queries, keys[:, :2], values[:, :2], source_keep[:, :2])
assert np.allclose(context, without_pad)

shape Q/K/weights: (1, 2, 2) (1, 3, 2) (1, 2, 3)
weights: [[[0.67, 0.33, 0.0], [0.33, 0.67, 0.0]]]
context có mask: [[[1.34, 0.66], [0.66, 1.34]]]
context quên mask: [[[98.493, 98.49], [98.49, 98.493]]]


Bảng trọng số có shape `(1, 2, 3)`: một câu, hai query đích, ba key nguồn. Số query và số key không cần bằng nhau, vì chuỗi đích và chuỗi nguồn có thể dài khác nhau. Query đầu đặt khoảng 0.67 vào key đầu và 0.33 vào key thứ hai, nên context là $0.67\cdot(2,0)+0.33\cdot(0,2)=(1.34,0.66)$. Query thứ hai thì ngược lại.

Nếu quên mask nguồn, key PAD có điểm rất cao, chiếm gần hết trọng số, và kéo context về gần `(99, 99)`. Causal mask của decoder không giúp được gì ở đây, vì nó chỉ che tương lai của chuỗi đích. Cross-attention cần một mask riêng cho PAD của chuỗi nguồn.

## Ví dụ 3: Học đảo hai ký hiệu, và chấm chuỗi model tự sinh

Có chín cặp ký hiệu tạo từ A, B, C. Đáp án là đảo ngược cặp, nên Python tính sẵn được. Bảy cặp đầu dùng để train. Hai cặp còn lại, `C B` và `C C`, được giữ riêng để thử những tổ hợp model chưa thấy. Model là encoder–decoder nhỏ dùng GRU, học từ đầu trên CPU. Trạng thái cuối của encoder được truyền sang làm trạng thái đầu cho decoder.

In [ ]:
import itertools
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(65)
PAD, BOS, EOS = 0, 1, 2
symbol_to_id = {"A": 3, "B": 4, "C": 5}
id_to_symbol = {value: key for key, value in symbol_to_id.items()}
pairs = list(itertools.product("ABC", repeat=2))
train_pairs, valid_pairs = pairs[:7], pairs[7:]

def make_batch(pairs):
    source = [[symbol_to_id[symbol] for symbol in pair] for pair in pairs]
    target = [row[::-1] for row in source]
    return (torch.tensor(source),
            torch.tensor([[BOS] + row for row in target]),
            torch.tensor([row + [EOS] for row in target]))

source, decoder_input, target = make_batch(train_pairs)
print("shape nguồn/input/label:", tuple(source.shape), tuple(decoder_input.shape), tuple(target.shape))
print("A B ->", source[1].tolist(), decoder_input[1].tolist(), target[1].tolist())
assert target[1].tolist() == [4, 3, EOS]

shape nguồn/input/label: (7, 2) (7, 3) (7, 3)
A B -> [3, 4] [1, 4, 3] [4, 3, 2]


Chuỗi nguồn có hai bước, chuỗi đích có ba bước vì thêm `EOS`. Với nguồn `A B`, decoder nhận `[BOS, B, A]` và phải đoán `[B, A, EOS]`. Bảng ký hiệu được định nghĩa từ trước, không học từ dữ liệu validation.

In [ ]:
class SmallSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(6, 12, padding_idx=PAD)
        self.encoder = nn.GRU(12, 24, batch_first=True)
        self.decoder = nn.GRU(12, 24, batch_first=True)
        self.output_layer = nn.Linear(24, 6)

    def forward(self, source, decoder_input):
        _, state = self.encoder(self.embedding(source))
        hidden, _ = self.decoder(self.embedding(decoder_input), state)
        return self.output_layer(hidden)

model = SmallSeq2Seq()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
history = []
for step in range(220):
    optimizer.zero_grad()
    logits = model(source, decoder_input)
    loss = criterion(logits.reshape(-1, 6), target.reshape(-1))
    loss.backward()
    optimizer.step()
    history.append(loss.item())

model.eval()
valid_source, valid_input, valid_target = make_batch(valid_pairs)
with torch.no_grad():
    train_logits = model(source, decoder_input)
    valid_logits = model(valid_source, valid_input)
    valid_loss = criterion(valid_logits.reshape(-1, 6), valid_target.reshape(-1))
print("logits train:", tuple(train_logits.shape))
print("loss train đầu/cuối:", round(history[0], 4), round(history[-1], 4))
print("loss validation teacher-forced:", round(valid_loss.item(), 4))
assert history[-1] < history[0]

logits train: (7, 3, 6)
loss train đầu/cuối: 1.8026 0.0003
loss validation teacher-forced: 1.2613


Logit có shape `(7, 3, 6)`: bảy cặp, ba bước, sáu token có thể đoán. Ta trải thành `(21, 6)` để tính cross-entropy, giữ nguyên trục sáu lớp. Loss train giảm từ 1.80 xuống gần 0, nhưng loss validation vẫn là 1.26.

Cả hai con số đều tính với teacher forcing, tức decoder luôn được đưa token đúng của bước trước. Khi dùng thật, decoder phải nhận lại chính token nó vừa đoán. Vì vậy ta cần chạy vòng sinh để kiểm.

In [ ]:
def generate(pair, max_steps=4):
    source = torch.tensor([[symbol_to_id[symbol] for symbol in pair]])
    result, ended = [], False
    with torch.no_grad():
        _, state = model.encoder(model.embedding(source))
        current = torch.tensor([[BOS]])
        for _ in range(max_steps):
            hidden, state = model.decoder(model.embedding(current), state)
            logits = model.output_layer(hidden[:, -1]).clone()
            logits[:, [PAD, BOS]] = -torch.inf  # PAD và BOS không bao giờ là token đầu ra hợp lệ.
            token = int(logits.argmax(dim=-1).item())
            if token == EOS:
                ended = True
                break
            result.append(id_to_symbol[token])
            current = torch.tensor([[token]])
    return tuple(result), ended

for name, split in [("train", train_pairs), ("validation", valid_pairs)]:
    correct = 0
    for pair in split:
        prediction, ended = generate(pair)
        gold = tuple(reversed(pair))
        correct += ended and prediction == gold
        if name == "validation":
            print("nguồn", " ".join(pair), "gold", " ".join(gold),
                  "sinh", " ".join(prediction), "EOS", ended)
    print(name, "exact match:", f"{correct}/{len(split)}")

train exact match: 7/7
nguồn C B gold B C sinh B C EOS True
nguồn C C gold C C sinh C A EOS True
validation exact match: 1/2


Vòng sinh bắt đầu từ `BOS`, mỗi bước lấy token có điểm cao nhất, rồi đưa token ấy làm đầu vào cho bước sau. Vòng dừng khi gặp `EOS`, hoặc khi hết `max_steps`. Giới hạn này giữ cho vòng lặp không chạy mãi nếu model không chịu dừng.

Model sinh đúng cả bảy cặp train. Trên validation, `C B` đúng, nhưng `C C` lại sinh ra `C A`: sai token thứ hai, dù vẫn dừng đúng bằng `EOS`. Exact match yêu cầu đúng cả chuỗi và có `EOS`, chứ không chỉ đúng token đầu. Loss train gần 0 chưa có nghĩa model đã học được quy tắc đảo ngược cho mọi chuỗi. Đáp án đúng được tính độc lập bằng `reversed(pair)`, không dựa vào model.